# Sentiment Analysis — `fact_customer_reviews`

Reproduces the sentiment-scoring pipeline used to build `fact_customer_reviews_with_sentiment`
from the raw `fact_customer_reviews` table.

Uses **VADER** (Valence Aware Dictionary and sEntiment Reasoner) from NLTK to score review text,
then combines that score with each review's star **Rating** to flag cases where the written
tone and the numeric rating disagree (e.g. a 2-star review that reads positively).

**Output columns added:** `SentimentScore`, `SentimentCategory`, `SentimentBucket`


In [ ]:
import pandas as pd
import re
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...


True

## 1. Load raw reviews

Replace the source below with wherever your `fact_customer_reviews` data actually lives.
In the original project this came from SQL Server (`PortfolioProject_MarketingAnalytics`),
queried before being handed to Python for sentiment scoring.


In [ ]:
# --- Option A: load from a CSV export of fact_customer_reviews ---
df = pd.read_excel('all_datasets.xlsx', sheet_name='fact_customer_reviews')
print(df.shape)
df.head()

(1363, 9)


,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText,SentimentScore,SentimentCategory,SentimentBucket
0,19,34,13,2024-01-04,4,The quality is top-notch.,0.0,Positive,0.0 to 0.49
1,22,52,3,2025-10-30,4,The quality is top-notch.,0.0,Positive,0.0 to 0.49
2,27,56,6,2025-11-24,4,The quality is top-notch.,0.0,Positive,0.0 to 0.49
3,32,91,1,2023-07-15,4,The quality is top-notch.,0.0,Positive,0.0 to 0.49
4,94,2,16,2025-03-14,4,The quality is top-notch.,0.0,Positive,0.0 to 0.49


## 2. Clean review text

Strips leading/trailing whitespace and collapses repeated internal spaces before scoring.


In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text) 
    return text

df['ReviewText'] = df['ReviewText'].apply(clean_text)
df[['ReviewText']].head()

,ReviewText
0,The quality is top-notch.
1,The quality is top-notch.
2,The quality is top-notch.
3,The quality is top-notch.
4,The quality is top-notch.


## 3. Score sentiment with VADER

VADER's `compound` score summarizes overall sentiment on a scale from **-1** (most negative)
to **+1** (most positive). This becomes `SentimentScore`.


In [5]:
sia = SentimentIntensityAnalyzer()

def get_compound_score(text):
    return sia.polarity_scores(text)['compound']

df['SentimentScore'] = df['ReviewText'].apply(get_compound_score).round(4)
df[['ReviewText', 'SentimentScore']].head()

,ReviewText,SentimentScore
0,The quality is top-notch.,0.0
1,The quality is top-notch.,0.0
2,The quality is top-notch.,0.0
3,The quality is top-notch.,0.0
4,The quality is top-notch.,0.0


## 4. Classify `SentimentCategory`

Combines the customer's star **Rating** with the **sign** of the VADER score, so a rating that
disagrees with the text tone gets flagged as "Mixed" rather than just trusting one signal:

| Rating | Score sign | Category |
|---|---|---|
| 1–2 | ≤ 0 | Negative |
| 1–2 | > 0 | Mixed Negative *(low rating, positive-sounding text)* |
| 3 | < 0 | Mixed Negative |
| 3 | = 0 | Neutral |
| 3 | > 0 | Mixed Positive |
| 4–5 | any | Positive |


In [6]:
def classify_sentiment(row):
    rating = row['Rating']
    score = row['SentimentScore']

    if rating <= 2:
        return 'Negative' if score <= 0 else 'Mixed Negative'
    elif rating == 3:
        if score < 0:
            return 'Mixed Negative'
        elif score == 0:
            return 'Neutral'
        else:
            return 'Mixed Positive'
    else:  # rating 4 or 5
        return 'Positive'

df['SentimentCategory'] = df.apply(classify_sentiment, axis=1)
df['SentimentCategory'].value_counts()

SentimentCategory
Positive          840
Negative          226
Mixed Negative    196
Mixed Positive     86
Neutral            15
Name: count, dtype: int64

## 5. Bucket the raw score

Groups the continuous `SentimentScore` into four broad bands, useful for slicers/visuals in Power BI.


In [7]:
def bucket_score(score):
    if score >= 0.5:
        return '0.5 to 1.0'
    elif score >= 0:
        return '0.0 to 0.49'
    elif score >= -0.5:
        return '-0.49 to 0.0'
    else:
        return '-1.0 to -0.5'

df['SentimentBucket'] = df['SentimentScore'].apply(bucket_score)
df['SentimentBucket'].value_counts()

SentimentBucket
0.0 to 0.49     564
0.5 to 1.0      464
-0.49 to 0.0    317
-1.0 to -0.5     18
Name: count, dtype: int64

## 6. Review the result


In [8]:
print(df.shape)
df[['ReviewID', 'Rating', 'ReviewText', 'SentimentScore', 'SentimentCategory', 'SentimentBucket']].head(10)

(1363, 9)


,ReviewID,Rating,ReviewText,SentimentScore,SentimentCategory,SentimentBucket
0,19,4,The quality is top-notch.,0.0,Positive,0.0 to 0.49
1,22,4,The quality is top-notch.,0.0,Positive,0.0 to 0.49
2,27,4,The quality is top-notch.,0.0,Positive,0.0 to 0.49
3,32,4,The quality is top-notch.,0.0,Positive,0.0 to 0.49
4,94,4,The quality is top-notch.,0.0,Positive,0.0 to 0.49
5,103,5,The quality is top-notch.,0.0,Positive,0.0 to 0.49
6,116,4,The quality is top-notch.,0.0,Positive,0.0 to 0.49
7,120,4,The quality is top-notch.,0.0,Positive,0.0 to 0.49
8,133,4,The quality is top-notch.,0.0,Positive,0.0 to 0.49
9,149,4,The quality is top-notch.,0.0,Positive,0.0 to 0.49


In [ ]:
df.to_csv('fact_customer_reviews_with_sentiment.csv', index=False)
print('Saved fact_customer_reviews_with_sentiment.csv')